# Day 59 — MultiIndex & Advanced GroupBy
**Month 3 | Week 6 | groupby transform · cumcount · named agg · multi-level index · groupby filter**

---

> **Real-world framing:**
>
> A single groupby gets you totals. Advanced groupby gets you answers.
>
> "What % of each region's revenue does Electronics account for?" — `transform`.
> "Which customers placed their 2nd or 3rd order this month?" — `cumcount`.
> "Give me mean, max, and count in one line with clean column names" — named agg.
> "Show me only categories where avg revenue > ₹3,000" — `groupby filter`.
>
> These are the four patterns that separate a junior analyst from a senior one.
> They appear in every real dashboard, every cohort analysis, every segmentation report.
> Master them and you can answer any "breakdown by group" question a client asks.

---

**Skills used today:** Python basics, Pandas (all prior Month 3 days), NumPy (Day 57)
**New today:** `groupby transform` · `groupby cumcount` · named agg (`agg` with dict) ·
`groupby filter` · `pd.MultiIndex` · `unstack` · `swaplevel` · `xs`

**Total: 80 pts + 10★ bonus**

---


---
## 📦 Section 1 — Raw Data (Read Only — Never Modify)

In [1]:
# ── RAW DATA — DO NOT MODIFY BELOW THIS CELL ──────────────────────────────
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

rng = np.random.default_rng(seed=59)
n = 400

regions    = ['North', 'South', 'East', 'West']
categories = ['Electronics', 'Clothing', 'Home', 'Sports', 'Books']
segments   = ['Regular', 'Premium', 'VIP']
statuses   = ['Delivered', 'Returned', 'Pending']

raw = pd.DataFrame({
    'order_id'    : [f'ORD{3000+i}' for i in range(n)],
    'customer_id' : [f'CUST{rng.integers(1001, 1051)}' for _ in range(n)],
    'order_date'  : pd.to_datetime(
        rng.choice(pd.date_range('2024-01-01', '2024-12-31'), n, replace=True)
    ),
    'region'      : rng.choice(regions,    n, p=[0.30, 0.25, 0.25, 0.20]),
    'category'    : rng.choice(categories, n, p=[0.30, 0.25, 0.20, 0.15, 0.10]),
    'units'       : rng.integers(1, 20, n),
    'unit_price'  : rng.uniform(50, 500, n).round(2),
    'discount_pct': rng.choice([0, 5, 10, 15, 20], n, p=[0.4, 0.25, 0.20, 0.10, 0.05]),
    'segment'     : rng.choice(segments,   n, p=[0.50, 0.35, 0.15]),
    'status'      : rng.choice(statuses,   n, p=[0.70, 0.20, 0.10]),
})
raw['revenue'] = (raw['units'] * raw['unit_price'] * (1 - raw['discount_pct']/100)).round(2)
df = raw.copy()

print(f"Shape: {df.shape}")
print(f"Unique customers: {df['customer_id'].nunique()}")
print(df[['order_id','customer_id','order_date','region','category','revenue']].head(6).to_string(index=False))


Shape: (400, 11)
Unique customers: 50
order_id customer_id order_date region    category  revenue
 ORD3000    CUST1040 2024-01-12  South    Clothing  5315.40
 ORD3001    CUST1030 2024-04-05  South Electronics  3879.50
 ORD3002    CUST1038 2024-06-24  South       Books   634.94
 ORD3003    CUST1042 2024-10-05   East Electronics  1635.40
 ORD3004    CUST1032 2024-11-23   East Electronics  2771.26
 ORD3005    CUST1006 2024-10-04   East        Home   741.20


---
## 📖 Section 2 — Concept Notes

### 1. `groupby transform` — Broadcast Group Stats Back to Original Rows

Standard `groupby.agg()` collapses rows → one row per group.
`transform` keeps the same number of rows, broadcasting the group stat back to every member.

```python
# Total revenue per region (aggregation — reduces rows)
df.groupby('region')['revenue'].sum()          # 4 rows

# Each row gets its region's total (transform — same shape as df)
df['region_total'] = df.groupby('region')['revenue'].transform('sum')   # 400 rows

# Now you can compute a within-group share in one line:
df['region_share_pct'] = df['revenue'] / df['region_total'] * 100
```

**Why it matters:** Percentage-of-total, deviation from group mean, group rank — all require
transform. Any time you need "each row compared to its group", use transform.

Common transform functions: `'sum'`, `'mean'`, `'max'`, `'min'`, `'rank'`, `'cumsum'`

---

### 2. `cumcount` — Sequential Numbering Within Groups

`groupby.cumcount()` assigns 0, 1, 2, … to each row within its group, in the current row order.

```python
# Sort by customer and date first — order matters
df = df.sort_values(['customer_id', 'order_date'])

# Assign order number within each customer (0-indexed)
df['order_seq'] = df.groupby('customer_id').cumcount()

# Convert to 1-indexed (more readable)
df['order_num'] = df['order_seq'] + 1

# Filter: only each customer's 2nd order
second_orders = df[df['order_num'] == 2]

# Filter: only repeat customers (placed more than 1 order)
repeat = df[df.groupby('customer_id')['order_id'].transform('count') > 1]
```

**Why it matters:** Cohort analysis, first-purchase vs repeat analysis, funnel tracking —
all start with sequencing orders within a customer group.

---

### 3. Named Aggregation — Clean Multi-Stat Summaries

Instead of `agg(['sum','mean','count'])` which gives ugly column names, use named kwargs:

```python
# Old way — ugly MultiIndex columns
df.groupby('region')['revenue'].agg(['sum', 'mean', 'count'])

# New way — named agg (Pandas 0.25+) — clean single-level columns
summary = df.groupby('region').agg(
    total_revenue  = ('revenue', 'sum'),
    avg_revenue    = ('revenue', 'mean'),
    order_count    = ('revenue', 'count'),
    max_revenue    = ('revenue', 'max'),
    unique_cats    = ('category', 'nunique')
).round(2)
```

**Multiple columns, multiple stats — one clean operation.**
Named agg is what goes into client reports — not raw MultiIndex column tuples.

---

### 4. `groupby filter` — Keep Only Groups Meeting a Condition

`groupby.filter(func)` drops entire groups where `func` returns False.

```python
# Keep only regions with more than 80 orders
df_filtered = df.groupby('region').filter(lambda g: len(g) > 80)

# Keep only categories where mean revenue > 3000
high_val = df.groupby('category').filter(lambda g: g['revenue'].mean() > 3000)

# Keep only customers who placed at least 3 orders
loyal = df.groupby('customer_id').filter(lambda g: len(g) >= 3)
```

**Unlike boolean masking**, `filter` acts on the whole group — either all rows of the group
are kept or all are dropped. Masking acts row by row.

---

### 5. MultiIndex — Two-Level Index from GroupBy

When you `groupby` multiple columns, the result has a MultiIndex:

```python
# Two-level groupby → MultiIndex result
multi = df.groupby(['region', 'category'])['revenue'].agg(['sum','mean']).round(2)
# Index has two levels: region (outer) and category (inner)

# Access a specific outer level
multi.loc['North']                    # all categories for North

# Access a specific combination
multi.loc[('North', 'Electronics')]   # North + Electronics row

# Cross-section: all regions for one category
multi.xs('Electronics', level='category')

# Reshape: pivot inner level to columns
multi_unstacked = multi.unstack(level='category')   # regions as rows, categories as cols

# Swap levels and sort
multi.swaplevel().sort_index()        # category outer, region inner
```

**`unstack` is the bridge between MultiIndex and a pivot table.**
Once you `unstack`, you have a regular 2D DataFrame you can style and export.

---

### Common Mistakes → Fixes

| Mistake | Fix |
|---------|-----|
| `transform` on wrong axis | Always apply to a single column Series, not full df |
| `cumcount` without sorting first | Sort by group key + date before `cumcount` |
| Named agg syntax error | `('column', 'function')` — column first, function second |
| `filter` vs `query` confusion | `filter` works on groups; `query` works on rows |
| `unstack` creates NaN | Some region+category combos have no data — fill with 0 |
| `xs` KeyError | Pass `level=` explicitly: `xs('Electronics', level='category')` |


---
## ✏️ Section 3 — Practice Tasks

Total = **80 pts + 10★ bonus**. Never modify raw data. Work on `df = raw.copy()`.

---
### Task A — groupby transform (20 pts)

#### A1 — Region Revenue Share (10 pts)
For each order, compute what **% of its region's total revenue** that order contributes.

Steps:
1. Use `transform('sum')` to add column `region_total_rev` — each row gets its region's total
2. Compute `revenue_share_pct = revenue / region_total_rev * 100`, rounded to 4dp
3. Add both columns to `df`

Print:
- The region totals (just the 4 unique values): `df.groupby('region')['region_total_rev'].first()`
- The top 5 orders by `revenue_share_pct` (columns: order_id, region, revenue, revenue_share_pct)
- One f-string: which single order has the largest region share? Format: `f"Order {id} contributes {pct:.2f}% of {region}'s total revenue"`

#### A2 — Deviation from Group Mean (10 pts)
Compute how much each order's revenue **deviates from its category's mean revenue**.

1. Add `category_mean_rev` using `transform('mean')`, rounded to 2dp
2. Add `rev_deviation = revenue - category_mean_rev`, rounded to 2dp
3. Add `deviation_pct = rev_deviation / category_mean_rev * 100`, rounded to 2dp

Print:
- Category means (5 values): `df.groupby('category')['category_mean_rev'].first().sort_values(ascending=False)`
- Top 3 orders most above their category mean (by `rev_deviation`, descending)
- Top 3 orders most below their category mean (by `rev_deviation`, ascending)
- One NRA: what does a large positive deviation in Electronics tell the sales team?


In [2]:
# A1 — Region Revenue Share (10 pts)
# transform('sum') → region_total_rev
# revenue / region_total_rev * 100 → revenue_share_pct
# Print region totals, top 5 by share, f-string for max share order
df['region_total_rev']  = df.groupby('region')['revenue'].transform('sum').round(2)
df['revenue_share_pct'] = (df['revenue'] / df['region_total_rev'] * 100).round(4)

print("Region totals:")
print(df.groupby('region')['region_total_rev'].first().sort_values(ascending=False))
print()
top5 = df.nlargest(5, 'revenue_share_pct')[['order_id','region','revenue','revenue_share_pct']]
print("Top 5 orders by region share:")
print(top5.to_string(index=False))
print()
top_row = df.loc[df['revenue_share_pct'].idxmax()]
print(f"Order {top_row['order_id']} contributes {top_row['revenue_share_pct']:.2f}% of {top_row['region']}'s total revenue")


Region totals:
region
North    356963.15
East     244930.55
South    234605.82
West     221061.82
Name: region_total_rev, dtype: float64

Top 5 orders by region share:
order_id region  revenue  revenue_share_pct
 ORD3119   West  7879.84             3.5645
 ORD3043   West  7830.43             3.5422
 ORD3263  South  8150.65             3.4742
 ORD3218   East  8272.98             3.3777
 ORD3247   West  7404.80             3.3497

Order ORD3119 contributes 3.56% of West's total revenue


In [3]:
# A2 — Deviation from Group Mean (10 pts)
# transform('mean') → category_mean_rev
# revenue - category_mean_rev → rev_deviation
# rev_deviation / category_mean_rev * 100 → deviation_pct
# Print category means, top 3 above + top 3 below, NRA
df['category_mean_rev'] = df.groupby('category')['revenue'].transform('mean').round(2)
df['rev_deviation']     = (df['revenue'] - df['category_mean_rev']).round(2)
df['deviation_pct']     = (df['rev_deviation'] / df['category_mean_rev'] * 100).round(2)

print("Category means (desc):")
print(df.groupby('category')['category_mean_rev'].first().sort_values(ascending=False))
print()
print("Top 3 most above category mean:")
print(df.nlargest(3,'rev_deviation')[['order_id','category','revenue','category_mean_rev','rev_deviation']].to_string(index=False))
print()
print("Top 3 most below category mean:")
print(df.nsmallest(3,'rev_deviation')[['order_id','category','revenue','category_mean_rev','rev_deviation']].to_string(index=False))


Category means (desc):
category
Sports         2772.20
Books          2767.25
Clothing       2644.96
Electronics    2621.21
Home           2508.01
Name: category_mean_rev, dtype: float64

Top 3 most above category mean:
order_id    category  revenue  category_mean_rev  rev_deviation
 ORD3214 Electronics  9175.10            2621.21        6553.89
 ORD3149    Clothing  9005.81            2644.96        6360.85
 ORD3108      Sports  8848.29            2772.20        6076.09

Top 3 most below category mean:
order_id    category  revenue  category_mean_rev  rev_deviation
 ORD3177       Books   144.01            2767.25       -2623.24
 ORD3382    Clothing    66.55            2644.96       -2578.41
 ORD3213 Electronics    71.61            2621.21       -2549.60


**NRA — Electronics deviation insight:**

**N:** Among Electronics orders, there is a wide spread of order revenues, with some reaching far above the category mean of ₹2,621.21 and others dipping well below.

**R:** A large positive deviation suggests that a small number of Electronics transactions are significantly high‑ticket items (bulk orders or expensive gadgets). These outliers pull up the average, while most orders may cluster around a lower amount.

**A:** The sales team should segment Electronics orders into “high‑value” and “standard” tiers, and develop separate engagement strategies. High‑deviation, high‑revenue orders could be candidates for dedicated account management or premium after‑sales support, while low‑deviation orders may respond to cross‑sell offers of accessories or warranties.

---
### Task B — cumcount & Sequential Analysis (20 pts)

#### B1 — Customer Order Sequence (10 pts)
Analyse the order history for each customer.

1. Sort `df` by `['customer_id', 'order_date']` (in-place or assign back)
2. Add `order_num` = `cumcount() + 1` per customer group (1-indexed)
3. Add `is_first_order` = True where `order_num == 1`, False otherwise

Print:
- Distribution of `order_num` values: `df['order_num'].value_counts().sort_index().head(10)`
- How many customers placed exactly 1 order vs 2+ orders? (use groupby + count, then compare)
- The 5 customers with the most orders (customer_id and order count)

#### B2 — First vs Repeat Order Revenue (10 pts)
Compare revenue behaviour between first orders and repeat orders.

1. Create `first_orders = df[df['order_num'] == 1]`
2. Create `repeat_orders = df[df['order_num'] > 1]`

Compute and print a comparison table:
```
                First Orders    Repeat Orders
Count:               xxx              xxx
Avg Revenue:       ₹x,xxx           ₹x,xxx
Total Revenue:  ₹xxx,xxx         ₹xxx,xxx
Median Revenue:    ₹x,xxx           ₹x,xxx
```

Then: which segment (Regular/Premium/VIP) has the highest avg revenue on **repeat** orders?
Print as f-string with computed values.


In [4]:
# B1 — Customer Order Sequence (10 pts)
# Sort by customer_id + order_date
# order_num = cumcount() + 1
# is_first_order = order_num == 1
# Print value_counts, 1-order vs 2+ customers, top 5 customers by orders
df = df.sort_values(['customer_id','order_date']).reset_index(drop=True)
df['order_num']      = df.groupby('customer_id').cumcount() + 1
df['is_first_order'] = df['order_num'] == 1

print("Order number distribution:")
print(df['order_num'].value_counts().sort_index().head(10))
print()

cust_counts = df.groupby('customer_id')['order_id'].count()
one_order  = (cust_counts == 1).sum()
multi_order = (cust_counts > 1).sum()
print(f"Customers with exactly 1 order: {one_order}")
print(f"Customers with 2+ orders:       {multi_order}")
print()
print("Top 5 customers by order count:")
print(cust_counts.nlargest(5).reset_index().rename(columns={'order_id':'order_count'}).to_string(index=False))


Order number distribution:
order_num
1     50
2     50
3     50
4     49
5     44
6     39
7     34
8     25
9     17
10    14
Name: count, dtype: int64

Customers with exactly 1 order: 0
Customers with 2+ orders:       50

Top 5 customers by order count:
customer_id  order_count
   CUST1001           15
   CUST1004           15
   CUST1006           15
   CUST1018           14
   CUST1010           12


In [5]:
# B2 — First vs Repeat Revenue Comparison (10 pts)
# first_orders = df[df['order_num'] == 1]
# repeat_orders = df[df['order_num'] > 1]
# Build comparison table
# Best segment on repeat orders
first_orders  = df[df['order_num'] == 1]
repeat_orders = df[df['order_num'] > 1]

f_cnt, r_cnt     = len(first_orders), len(repeat_orders)
f_avg, r_avg     = first_orders['revenue'].mean(), repeat_orders['revenue'].mean()
f_tot, r_tot     = first_orders['revenue'].sum(), repeat_orders['revenue'].sum()
f_med, r_med     = first_orders['revenue'].median(), repeat_orders['revenue'].median()

print(f"{'':20} {'First Orders':>15} {'Repeat Orders':>15}")
print("-"*52)
print(f"{'Count:':<20} {f_cnt:>15,} {r_cnt:>15,}")
print(f"{'Avg Revenue:':<20} ₹{f_avg:>14,.2f} ₹{r_avg:>14,.2f}")
print(f"{'Total Revenue:':<20} ₹{f_tot:>14,.2f} ₹{r_tot:>14,.2f}")
print(f"{'Median Revenue:':<20} ₹{f_med:>14,.2f} ₹{r_med:>14,.2f}")
print()

seg_repeat = repeat_orders.groupby('segment')['revenue'].mean().round(2)
best_seg = seg_repeat.idxmax()
print(f"Highest avg revenue on repeat orders: {best_seg} segment — ₹{seg_repeat.max():,.2f}")


                        First Orders   Repeat Orders
----------------------------------------------------
Count:                            50             350
Avg Revenue:         ₹      2,830.38 ₹      2,617.26
Total Revenue:       ₹    141,518.76 ₹    916,042.58
Median Revenue:      ₹      2,217.22 ₹      2,148.59

Highest avg revenue on repeat orders: Regular segment — ₹2,874.58


---
### Task C — Named Aggregation (20 pts)

#### C1 — Region × Segment Summary (10 pts)
Build a full summary grouped by `['region', 'segment']` using named agg:

```python
summary = df.groupby(['region', 'segment']).agg(
    total_revenue   = ('revenue', 'sum'),
    avg_revenue     = ('revenue', 'mean'),
    order_count     = ('order_id', 'count'),
    max_single_order= ('revenue', 'max'),
    unique_categories= ('category', 'nunique')
).round(2)
```

Print the full summary table (should be 12 rows: 4 regions × 3 segments).

Then:
- Which region+segment combination has the highest total revenue? (use `idxmax()` on `total_revenue`)
- Which has the highest avg revenue per order?
- Print both as f-strings.

#### C2 — Category Performance Table (10 pts)
Build a category-level summary with named agg:

| Column | Source | Function |
|--------|--------|----------|
| `total_revenue` | revenue | sum |
| `avg_revenue` | revenue | mean |
| `order_count` | order_id | count |
| `return_count` | status | lambda: count of 'Returned' |
| `return_rate_pct` | computed | return_count / order_count * 100 |

> **Hint for return_count:** Use a lambda in agg:
> `return_count = ('status', lambda x: (x == 'Returned').sum())`

Round all numeric columns to 2dp. Sort by `total_revenue` descending.

Print the table. Then write one NRA on what the return rate reveals about product-category quality.


In [6]:
# C1 — Region × Segment Named Agg (10 pts)
# summary = df.groupby(['region','segment']).agg(
#     total_revenue=('revenue','sum'), avg_revenue=('revenue','mean'),
#     order_count=('order_id','count'), max_single_order=('revenue','max'),
#     unique_categories=('category','nunique')
# ).round(2)
# Print full table
# Best by total revenue and best by avg revenue — both dynamic
summary = df.groupby(['region','segment']).agg(
    total_revenue    = ('revenue','sum'),
    avg_revenue      = ('revenue','mean'),
    order_count      = ('order_id','count'),
    max_single_order = ('revenue','max'),
    unique_categories= ('category','nunique')
).round(2)

print(summary.to_string())
print()
best_total = summary['total_revenue'].idxmax()
best_avg   = summary['avg_revenue'].idxmax()
print(f"Highest total revenue: {best_total[0]} / {best_total[1]} — ₹{summary.loc[best_total,'total_revenue']:,.2f}")
print(f"Highest avg revenue:   {best_avg[0]} / {best_avg[1]} — ₹{summary.loc[best_avg,'avg_revenue']:,.2f}")


                total_revenue  avg_revenue  order_count  max_single_order  unique_categories
region segment                                                                              
East   Premium       93499.83      2460.52           38           6908.98                  5
       Regular      120843.89      3021.10           40           8272.98                  5
       VIP           30586.83      2184.77           14           5699.77                  5
North  Premium      135203.23      2759.25           49           8848.29                  5
       Regular      179903.44      2855.61           63           9175.10                  5
       VIP           41856.48      2989.75           14           9005.81                  4
South  Premium       87085.68      2638.96           33           7225.60                  5
       Regular      130652.70      2512.55           52           8150.65                  5
       VIP           16867.44      1124.50           15           3855

In [7]:
# C2 — Category Performance Table with Return Rate (10 pts)
# Named agg with lambda for return_count
# Compute return_rate_pct after agg
# Sort by total_revenue desc, print
# NRA on return rate below
cat_summary = df.groupby('category').agg(
    total_revenue = ('revenue','sum'),
    avg_revenue   = ('revenue','mean'),
    order_count   = ('order_id','count'),
    return_count  = ('status', lambda x: (x == 'Returned').sum())
).round(2)
cat_summary['return_rate_pct'] = (cat_summary['return_count'] / cat_summary['order_count'] * 100).round(2)
cat_summary = cat_summary.sort_values('total_revenue', ascending=False)
print(cat_summary.to_string())


             total_revenue  avg_revenue  order_count  return_count  return_rate_pct
category                                                                           
Electronics      285711.45      2621.21          109            20            18.35
Clothing         275075.34      2644.96          104            19            18.27
Home             203149.15      2508.01           81            19            23.46
Sports           166331.99      2772.20           60            11            18.33
Books            127293.41      2767.25           46             9            19.57


**NRA — Return rate insight:**

**N:** Home goods show the highest return rate at 23.46%, while Clothing and Electronics are around 18–19%, and Books and Sports just under 20%.

**R:** A high return rate in Home goods may indicate issues with product quality, inaccurate online descriptions, sizing/fit problems, or fragile items arriving damaged. Returns erode the profitability of what is otherwise a solid revenue category.

**A:** A root‑cause analysis should be launched for Home returns — examining top‑returned SKUs, customer feedback, and packaging quality. Insights can then feed into product page improvements, quality control reviews, and potentially a revised returns policy for high‑return categories.


---
### Task D — MultiIndex Operations (20 pts)

#### D1 — Build and Navigate a MultiIndex (10 pts)
Create a MultiIndex summary grouped by `['region', 'category']`:

```python
multi = df.groupby(['region', 'category']).agg(
    total_revenue = ('revenue', 'sum'),
    order_count   = ('order_id', 'count'),
    avg_revenue   = ('revenue', 'mean')
).round(2)
```

This produces a 20-row DataFrame (4 regions × 5 categories) with a 2-level index.

Demonstrate all four access patterns:
1. `multi.loc['North']` — all categories for North region
2. `multi.loc[('East', 'Electronics')]` — specific combination
3. `multi.xs('Electronics', level='category')` — all regions for Electronics
4. `multi.xs('VIP', level='segment')` ← **this will fail** — fix it by using `multi.xs('South', level='region')` instead

Print each result with a label.

#### D2 — Unstack and Reshape (10 pts)
From your `multi` DataFrame above:

1. **Unstack** the `category` level to get a wide DataFrame:
   `wide = multi['total_revenue'].unstack(level='category').fillna(0).round(2)`
   This should have 4 rows (regions) and 5 columns (categories).

2. Print `wide`.

3. Add a `row_total` column = sum across all categories (axis=1).
   Add a `top_category` column = name of the category with highest revenue per region
   (use `.idxmax(axis=1)` on the 5 category columns only).

4. Print the final table with `row_total` and `top_category` columns.

5. One f-string per region: `f"{region}: ₹{total:,.2f} total — top category: {cat}"`


In [8]:
# D1 — Build and Navigate MultiIndex (10 pts)
# multi = df.groupby(['region','category']).agg(...).round(2)
# Show: multi.loc['North']
# Show: multi.loc[('East','Electronics')]
# Show: multi.xs('Electronics', level='category')
# Show: multi.xs('South', level='region')
multi = df.groupby(['region','category']).agg(
    total_revenue = ('revenue','sum'),
    order_count   = ('order_id','count'),
    avg_revenue   = ('revenue','mean')
).round(2)

print("1. multi.loc['North']:")
print(multi.loc['North'])
print()
print("2. multi.loc[('East','Electronics')]:")
print(multi.loc[('East','Electronics')])
print()
print("3. multi.xs('Electronics', level='category'):")
print(multi.xs('Electronics', level='category'))
print()
print("4. multi.xs('South', level='region'):")
print(multi.xs('South', level='region'))


1. multi.loc['North']:
             total_revenue  order_count  avg_revenue
category                                            
Books             44068.27           15      2937.88
Clothing          76672.67           26      2948.95
Electronics       92934.54           32      2904.20
Home              79150.69           31      2553.25
Sports            64136.98           22      2915.32

2. multi.loc[('East','Electronics')]:
total_revenue    71876.30
order_count         28.00
avg_revenue       2567.01
Name: (East, Electronics), dtype: float64

3. multi.xs('Electronics', level='category'):
        total_revenue  order_count  avg_revenue
region                                         
East         71876.30           28      2567.01
North        92934.54           32      2904.20
South        62843.32           28      2244.40
West         58057.29           21      2764.63

4. multi.xs('South', level='region'):
             total_revenue  order_count  avg_revenue
category            

In [9]:
# D2 — Unstack and Reshape (10 pts)
# wide = multi['total_revenue'].unstack('category').fillna(0).round(2)
# Print wide
# Add row_total and top_category
# Print final table
# f-string per region
wide = multi['total_revenue'].unstack(level='category').fillna(0).round(2)
print("Wide (unstacked) table:")
print(wide.to_string())
print()

cat_cols = wide.columns.tolist()
wide['row_total']    = wide[cat_cols].sum(axis=1).round(2)
wide['top_category'] = wide[cat_cols].idxmax(axis=1)

print("With row totals and top category:")
print(wide.to_string())
print()

for region in wide.index:
    print(f"{region}: ₹{wide.loc[region,'row_total']:,.2f} total — top category: {wide.loc[region,'top_category']}")


Wide (unstacked) table:
category     Books  Clothing  Electronics      Home    Sports
region                                                       
East      24119.62  83930.29     71876.30  53316.76  11687.58
North     44068.27  76672.67     92934.54  79150.69  64136.98
South     24394.18  58445.62     62843.32  46376.56  42546.14
West      34711.34  56026.76     58057.29  24305.14  47961.29

With row totals and top category:
category     Books  Clothing  Electronics      Home    Sports  row_total top_category
region                                                                               
East      24119.62  83930.29     71876.30  53316.76  11687.58  244930.55     Clothing
North     44068.27  76672.67     92934.54  79150.69  64136.98  356963.15  Electronics
South     24394.18  58445.62     62843.32  46376.56  42546.14  234605.82  Electronics
West      34711.34  56026.76     58057.29  24305.14  47961.29  221061.82  Electronics

East: ₹244,930.55 total — top category: Clothing
Nor

---
## 📊 Section 4 — Scoring Rubric

| Task | Sub-Task | Pts | What Is Checked |
|------|----------|-----|-----------------|
| **A — transform** | A1 Region share | 10 | `transform('sum')` used, share_pct computed and rounded, region totals printed, max-share f-string dynamic |
| | A2 Deviation from mean | 10 | `transform('mean')`, deviation + deviation_pct computed, top 3 above/below printed, NRA present |
| **B — cumcount** | B1 Order sequence | 10 | Sorted before cumcount, `+1` for 1-indexed, is_first_order column, value_counts, 1-order vs 2+ count, top 5 customers |
| | B2 First vs repeat | 10 | first/repeat split correct, 4-metric comparison table, best segment on repeat orders dynamic |
| **C — Named agg** | C1 Region × Segment | 10 | All 5 named metrics, 12-row result, best by total and avg dynamic (idxmax on MultiIndex) |
| | C2 Category + return rate | 10 | Lambda for return_count, return_rate_pct computed post-agg, sorted desc, NRA present |
| **D — MultiIndex** | D1 Build + navigate | 10 | All 4 access patterns demonstrated, correct results printed with labels |
| | D2 Unstack + reshape | 10 | `unstack('category')`, fillna(0), row_total and top_category columns, f-string per region |
| **★ Bonus** | See below | 10★ | — |
| **TOTAL** | | **80 + 10★** | |

---

### ⭐ Bonus Task — Advanced GroupBy Patterns (10★ pts)

#### Part 1 — groupby filter (4★)
Use `groupby.filter()` to answer two questions:

1. Keep only **categories** where the mean revenue across all their orders is above the overall dataset mean revenue.
   Print the kept categories and their mean revenue.

2. Keep only **customers** who placed at least **5 orders**.
   How many such customers exist? What is their combined total revenue vs the full dataset?
   Print a comparison: loyal customer revenue share as a % of total.

#### Part 2 — groupby rank + transform combo (6★)
Within each **region**, rank every order by revenue (rank 1 = highest in that region).
Use `df.groupby('region')['revenue'].rank(ascending=False, method='dense')`.
Add as column `region_rank`.

Then:
- Filter to only the **top 3 revenue orders per region** (`region_rank <= 3`)
- Print those 12 rows (3 per region × 4 regions): order_id, region, category, revenue, region_rank
- For each region, what category does the #1 order belong to? Print as f-string per region.


In [10]:
# ⭐ BONUS — Part 1: groupby filter (4★)

# Part 1a: categories where mean revenue > overall mean
overall_mean = df['revenue'].mean()
high_cat = df.groupby('category').filter(lambda g: g['revenue'].mean() > overall_mean)

print("Categories with mean revenue above overall mean:")
cat_means = high_cat.groupby('category')['revenue'].mean().round(2)
print(cat_means)
print()

# Part 1b: customers with >= 5 orders
loyal = df.groupby('customer_id').filter(lambda g: len(g) >= 5)
loyal_cust_count = loyal['customer_id'].nunique()
loyal_rev = loyal['revenue'].sum()
total_rev = df['revenue'].sum()
loyal_share = (loyal_rev / total_rev) * 100

print(f"Loyal customers (>=5 orders): {loyal_cust_count}")
print(f"Loyal customer total revenue: ₹{loyal_rev:,.2f}")
print(f"Full dataset total revenue: ₹{total_rev:,.2f}")
print(f"Loyal customer revenue share: {loyal_share:.2f}%")

Categories with mean revenue above overall mean:
category
Books       2767.25
Clothing    2644.96
Sports      2772.20
Name: revenue, dtype: float64

Loyal customers (>=5 orders): 44
Loyal customer total revenue: ₹972,734.61
Full dataset total revenue: ₹1,057,561.34
Loyal customer revenue share: 91.98%


In [11]:
# ⭐ BONUS — Part 2: region rank + top 3 (6★)

# Dense rank within each region (1 = highest revenue)
df['region_rank'] = df.groupby('region')['revenue'].rank(ascending=False, method='dense').astype(int)

# Top 3 per region
top3 = df[df['region_rank'] <= 3]
top3_display = top3[['order_id', 'region', 'category', 'revenue', 'region_rank']].sort_values(['region', 'region_rank'])

print("Top 3 revenue orders per region:")
print(top3_display.to_string(index=False))
print()

# Category of the #1 order in each region
rank1 = top3[top3['region_rank'] == 1]
for _, row in rank1.iterrows():
    print(f"{row['region']}: the #1 order is in {row['category']} (₹{row['revenue']:,.2f})")

Top 3 revenue orders per region:
order_id region    category  revenue  region_rank
 ORD3218   East       Books  8272.98            1
 ORD3230   East    Clothing  6908.98            2
 ORD3215   East    Clothing  6562.85            3
 ORD3214  North Electronics  9175.10            1
 ORD3149  North    Clothing  9005.81            2
 ORD3108  North      Sports  8848.29            3
 ORD3263  South      Sports  8150.65            1
 ORD3059  South    Clothing  7377.80            2
 ORD3389  South      Sports  7225.60            3
 ORD3119   West    Clothing  7879.84            1
 ORD3043   West Electronics  7830.43            2
 ORD3247   West      Sports  7404.80            3

North: the #1 order is in Electronics (₹9,175.10)
West: the #1 order is in Clothing (₹7,879.84)
East: the #1 order is in Books (₹8,272.98)
South: the #1 order is in Sports (₹8,150.65)


---

### Key Takeaway
`transform` is the most underused Pandas method in junior analyst work. Every time you see a senior analyst compute "what % of the group does this row account for" or "how far above the average is this?" — that's transform. It keeps the full row count so you can compare individuals to their group without a merge or a manual broadcast. Once you stop thinking in aggregations and start thinking in transforms, your analysis moves from "summary tables" to "row-level intelligence."

### Interview Frame
> "When a client says 'show me which orders are pulling above their category average', I use groupby transform to broadcast the category mean to every row, subtract to get the deviation, and filter on that column. The transform keeps the original 400-row DataFrame intact so the output can slot directly into any downstream model or dashboard filter — no merge, no index gymnastics."
